# 🔬 반도체 전문가 에이전트 — Colab 실행 환경

**런타임 설정**: 런타임 → 런타임 유형 변경 → GPU (T4 권장)

순서:
1. 📦 패키지 설치
2. 📁 프로젝트 마운트 / 업로드
3. 🔑 환경변수 설정
4. 🧠 BGE-M3 인덱싱 실행
5. 🔍 Hybrid Search 테스트

In [ ]:
# ── 셀 1: GPU 확인 ────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✓ GPU:', result.stdout.strip())
else:
    print('⚠ GPU 없음 — CPU로 실행 (느림). 런타임 → GPU로 전환 권장')

In [ ]:
# ── 셀 2: 패키지 설치 ─────────────────────────────────────────────────────────
# FlagEmbedding(BGE-M3) + Qdrant + Postgres 클라이언트
!pip install -q \
    'FlagEmbedding>=1.2' \
    'qdrant-client>=1.9,<2' \
    'psycopg[binary]>=3.1,<4' \
    'psycopg2-binary>=2.9' \
    'python-dotenv>=1.0' \
    'neo4j>=5.0,<6' \
    'APScheduler>=3.10,<4' \
    'httpx>=0.27' \
    'feedparser>=6.0' \
    'pypdf>=4.0' \
    'PyMuPDF>=1.24' \
    'Pillow>=10.0' \
    'pytesseract>=0.3' \
    'lxml>=5.0'
print('✓ 패키지 설치 완료')

In [ ]:
# ── 셀 3: 프로젝트 폴더 설정 ─────────────────────────────────────────────────
# 방법 A: Google Drive 마운트 (권장 — 파일이 Drive에 있는 경우)
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ★ 여기를 Drive 안의 실제 경로로 수정하세요
PROJECT_DIR = '/content/drive/MyDrive/semiconductor_agent_design'

# 경로 확인
if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f'프로젝트 폴더를 찾을 수 없음: {PROJECT_DIR}\n'
                            '위 경로를 실제 위치로 수정하세요.')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'✓ 프로젝트 경로: {os.getcwd()}')
print(f'  파일 목록: {os.listdir(".")[:10]}')

In [ ]:
# ── 셀 3-B (선택): Drive 대신 ZIP 업로드 방식 ───────────────────────────────
# Drive 마운트 없이 ZIP으로 업로드하려면 셀 3 대신 이 셀 실행

# from google.colab import files
# import zipfile, os, sys
#
# print('ZIP 파일 업로드 중... (.venv311 폴더 제외하고 압축하세요)')
# uploaded = files.upload()   # semiconductor_agent_design.zip 선택
# fname = list(uploaded.keys())[0]
# with zipfile.ZipFile(fname, 'r') as z:
#     z.extractall('/content/')
#
# PROJECT_DIR = '/content/semiconductor_agent_design'
# os.chdir(PROJECT_DIR)
# sys.path.insert(0, PROJECT_DIR)
# print(f'✓ 경로: {os.getcwd()}')

In [ ]:
# ── 셀 4: 환경변수 설정 ───────────────────────────────────────────────────────
# .env 파일이 프로젝트 폴더에 있으면 자동 로드됩니다.
# 없으면 아래에서 직접 설정하세요.

import os
from dotenv import load_dotenv

# .env 파일 로드
env_path = os.path.join(PROJECT_DIR, '.env')
if os.path.exists(env_path):
    load_dotenv(env_path)
    print('✓ .env 파일 로드 완료')
else:
    print('⚠ .env 파일 없음 — 아래 값들을 직접 설정하세요')

# GPU 사용 설정 (Colab T4 기준)
os.environ['BGE_DEVICE']       = 'cuda'   # GPU
os.environ['INDEXER_BATCH_SIZE'] = '64'   # GPU에서 배치 크게 (CPU면 8로)
os.environ['BGE_BATCH_SIZE']   = '64'

# 연결 정보 확인
print(f'  POSTGRES_DSN: {os.environ.get("POSTGRES_DSN", "NOT SET")[:40]}...')
print(f'  QDRANT_URL:   {os.environ.get("QDRANT_URL", "NOT SET")[:40]}...')
print(f'  BGE_DEVICE:   {os.environ.get("BGE_DEVICE")}')
print(f'  BATCH_SIZE:   {os.environ.get("INDEXER_BATCH_SIZE")}')

In [ ]:
# ── 셀 5: DB 연결 테스트 ──────────────────────────────────────────────────────
from app.db.postgres import get_pg_conn
from app.db.qdrant import get_qdrant_client

# Postgres
try:
    with get_pg_conn() as conn:
        with conn.cursor() as cur:
            cur.execute('SELECT COUNT(*) FROM tech_document_chunks')
            n = cur.fetchone()[0]
    print(f'✓ Postgres OK — tech_document_chunks: {n:,}행')
except Exception as e:
    print(f'✗ Postgres 연결 실패: {e}')

# Qdrant
try:
    qc = get_qdrant_client()
    colls = [c.name for c in qc.get_collections().collections]
    print(f'✓ Qdrant OK — 컬렉션: {colls}')
except Exception as e:
    print(f'✗ Qdrant 연결 실패: {e}')

In [ ]:
# ── 셀 6: BGE-M3 모델 로드 테스트 ────────────────────────────────────────────
# 첫 실행: 약 2.3GB 다운로드 (HuggingFace 캐시, 재실행 시 스킵)
import time
from app.rag.embedder import get_embedder, encode_single

t0 = time.time()
model = get_embedder()
print(f'✓ BGE-M3 로드 완료 ({time.time()-t0:.1f}s)')

# 샘플 인코딩
dense, sparse = encode_single('HBM3 thermal interface resistance copper pillar bonding')
print(f'  dense dim:     {len(dense)}')
print(f'  sparse tokens: {len(sparse)}')
print(f'  sparse top-5:  {sorted(sparse.items(), key=lambda x: -x[1])[:5]}')

In [ ]:
# ── 셀 7: semi_knowledge 컬렉션 생성 ─────────────────────────────────────────
from app.db.qdrant import ensure_semi_knowledge, get_qdrant_client, SEMI_KNOWLEDGE_COLLECTION

ensure_semi_knowledge()

info = get_qdrant_client().get_collection(SEMI_KNOWLEDGE_COLLECTION)
print(f'✓ semi_knowledge 컬렉션 준비')
print(f'  현재 포인트 수: {info.points_count or 0}')
print(f'  dense dim: {info.config.params.vectors["dense"].size}')
print(f'  sparse: {list(info.config.params.sparse_vectors.keys())}')

In [ ]:
# ── 셀 8: 전체 재인덱싱 실행 ─────────────────────────────────────────────────
# Postgres 청크 전체를 BGE-M3로 임베딩 → Qdrant 업서트
# T4 GPU 기준: ~2,000청크 약 3~5분
import logging, time
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

from app.rag.indexer import index_all_chunks

t0 = time.time()
n = index_all_chunks()
elapsed = time.time() - t0

print(f'\n✓ 인덱싱 완료!')
print(f'  처리 청크: {n:,}개')
print(f'  소요 시간: {elapsed:.1f}s')
print(f'  속도:      {n/elapsed:.1f} chunks/sec')

In [ ]:
# ── 셀 9: 인덱싱 결과 확인 ───────────────────────────────────────────────────
from app.db.qdrant import get_qdrant_client, SEMI_KNOWLEDGE_COLLECTION

info = get_qdrant_client().get_collection(SEMI_KNOWLEDGE_COLLECTION)
print(f'✓ semi_knowledge 최종 상태')
print(f'  총 포인트: {info.points_count:,}개')

# payload 분포 샘플
from app.db.qdrant import get_qdrant_client
client = get_qdrant_client()
sample, _ = client.scroll(
    collection_name=SEMI_KNOWLEDGE_COLLECTION,
    limit=5,
    with_payload=True,
    with_vectors=False,
)
print('\n샘플 payload:')
for p in sample:
    pl = p.payload
    print(f'  [{pl.get("source_type")}] {pl.get("source")} | {pl.get("domain")} | {pl.get("year")} | {pl.get("title","")[:50]}')

In [ ]:
# ── 셀 10: Hybrid Search 테스트 ───────────────────────────────────────────────
from app.rag.retriever import search, search_multi_query, get_context_for_llm

def show(results, label, n=5):
    print(f'\n{"─"*65}')
    print(f'  {label}')
    print(f'{"─"*65}')
    for i, r in enumerate(results[:n], 1):
        print(f'[{i}] score={r["score"]:.4f}  {r.get("source_type")} | {r.get("domain")} | {r.get("year")}')
        print(f'    {r.get("title","")[:70]}')
        print(f'    {r.get("text","")[:180].replace(chr(10)," ")}...')

# 테스트 1: HBM 열 설계
r1 = search('HBM3 thermal resistance copper pillar micro bump', top_k=5)
show(r1, '① HBM3 열 저항 — 전체 검색')

# 테스트 2: 논문만 필터
r2 = search('3D NAND flash endurance retention', top_k=5, filter_source_type='paper')
show(r2, '② 3D NAND 논문만 (source_type=paper)')

# 테스트 3: 도메인 필터
r3 = search('EUV stochastic defect overlay error', top_k=5, filter_domain='litho')
show(r3, '③ EUV (domain=litho)')

# 테스트 4: 멀티쿼리 퓨전
r4 = search_multi_query([
    'HBM CoWoS interposer bandwidth',
    'HBM thermal design power cooling',
    'HBM3E SK Hynix roadmap',
], top_k=5)
show(r4, '④ HBM 멀티쿼리 RRF 퓨전')

In [ ]:
# ── 셀 11: LLM 컨텍스트 생성 테스트 ──────────────────────────────────────────
from app.rag.retriever import get_context_for_llm

query = 'SK Hynix HBM4 advanced packaging 경쟁력 분석'
context = get_context_for_llm(
    query,
    top_k=6,
    max_chars=4000,
    filter_company='000660',   # SK Hynix
)

print(f'Query: {query}')
print(f'컨텍스트 길이: {len(context)}자')
print('─'*60)
print(context[:2000])

In [ ]:
# ── 셀 12: 커스텀 쿼리 ────────────────────────────────────────────────────────
from app.rag.retriever import search

my_query = 'GAA nanosheet transistor SRAM yield'  # ← 원하는 쿼리로 변경

results = search(
    my_query,
    top_k=10,
    # filter_source_type='paper',  # 논문만
    # filter_domain='logic',       # GAA = logic 도메인
    # filter_year_min=2023,        # 최근 논문만
)

print(f'Query: "{my_query}"  →  {len(results)} results\n')
for i, r in enumerate(results, 1):
    print(f'[{i}] {r["score"]:.4f}  [{r.get("source_type")}] {r.get("source")} {r.get("year")}')
    print(f'     {r.get("title","")[:80]}')
    print(f'     {r.get("text","")[:200].replace(chr(10)," ")}\n')